# OpenPlaque — Left-Coronary Backbone Branch Discovery v1
Label-neutral source-CCTA side-branch discovery after through-vessel continuity confirmation. Research use only; frozen master is never modified.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json, os, shutil, sys
DRIVE_ROOT = Path('/content/drive/MyDrive/OpenPlaque')
OUTPUT = DRIVE_ROOT / 'Left_Coronary_Backbone_Branch_Discovery_v1'
OUTPUT.mkdir(parents=True, exist_ok=True)
BRANCH = 'left-coronary-backbone-branch-discovery-from-main'
BASELINE = '0593b453959f5a353d644267fbeef24b514ef4d7'
SCIENCE_PIN = 'fde79ad8fe3cd463e8e623070fae0272b5529887'
REUSE_VALID_SOURCE_CACHE = True
(OUTPUT/'notebook_started.json').write_text(json.dumps({'status':'STARTED','branch':BRANCH,'science_pin':SCIENCE_PIN}, indent=2))
print('Output:', OUTPUT)
print('Science pin:', SCIENCE_PIN)

In [ ]:
!rm -rf /content/OpenPlaque
!git clone -q --branch {BRANCH} https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
!git -C /content/OpenPlaque checkout -q {SCIENCE_PIN}
!git -C /content/OpenPlaque rev-parse HEAD
!git -C /content/OpenPlaque merge-base HEAD {BASELINE}
!test "$(git -C /content/OpenPlaque rev-parse HEAD)" = "{SCIENCE_PIN}"
!test "$(git -C /content/OpenPlaque merge-base HEAD {BASELINE})" = "{BASELINE}"


In [ ]:
!pip install -q numpy pandas scipy matplotlib SimpleITK pytest
!pip install -q /content/OpenPlaque


In [ ]:
# Purge stale package modules before tests/science.
for name in list(sys.modules):
    if name == 'openplaque' or name.startswith('openplaque.'):
        del sys.modules[name]
!python -m py_compile /content/OpenPlaque/src/openplaque/left_coronary_backbone_branch_discovery_v1.py
!python -m pytest -q /content/OpenPlaque/tests/test_left_coronary_backbone_branch_discovery_v1.py
from openplaque.left_coronary_backbone_branch_discovery_v1 import synthetic_branch_discovery_self_test
print(synthetic_branch_discovery_self_test())


In [ ]:
required = [
    DRIVE_ROOT/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.npy',
    DRIVE_ROOT/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.json',
    DRIVE_ROOT/'Cache/Secondary_3D_Vesselness_Topology_v1/vesselness.npy',
    DRIVE_ROOT/'Cache/Master_Coronary_Anatomy_Baseline_v2/master_anatomy_summary.json',
    DRIVE_ROOT/'Cache/LAD_Frozen_Proximal_Reacquisition_v1/combined_lad_centerline.csv',
    DRIVE_ROOT/'Left_Proximal_Trunk_Continuation_QC_v1/accepted_proximal_trunk_continuation_candidate.csv',
    DRIVE_ROOT/'Joint_Three_Vessel_Template_Classifier_v1/candidate_04_source_path.csv',
    DRIVE_ROOT/'LCX_Distal_Reacquisition_v1_fixed/C7_extended_path.csv',
    DRIVE_ROOT/'Left_Coronary_Through_Vessel_Continuity_v1/summary.json',
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError('Missing required inputs:\n' + '\n'.join(missing))
(OUTPUT/'preflight_complete.json').write_text(json.dumps({'status':'PASS','required_count':len(required)}, indent=2))
print('Preflight PASS:', len(required), 'inputs')


In [ ]:
from openplaque.left_coronary_backbone_branch_discovery_v1 import run
result = run(drive_root=str(DRIVE_ROOT), output_dir=str(OUTPUT))
print(json.dumps(result['summary'], indent=2, default=str))
print('Report:', result['report'])
print('ZIP:', result['zip'])


In [ ]:
summary = json.loads((OUTPUT/'summary.json').read_text())
print('STATUS:', summary.get('status'))
print('C7 control:', summary.get('control',{}).get('pass'))
print('Backbone length mm:', summary.get('backbone_length_mm'))
print('Blind seeds:', summary.get('blind_seed_count'))
print('Valid branch hypotheses:', summary.get('valid_branch_hypothesis_count'))
print('Clustered additional branches:', summary.get('clustered_additional_branch_count'))
